# Assignment 8 — Handwritten Digit Recognition using ANN
**Dataset:** MNIST Handwritten Digits (loaded automatically via `tensorflow.keras.datasets.mnist`, no manual download needed)

**Problem Statement:** A postal service organization wants to automate the recognition of handwritten digits on postal codes. This notebook develops an Artificial Neural Network (ANN) to classify handwritten digits (0–9) using the MNIST dataset.


In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow version:", tf.__version__)


## Task 1: Data Understanding (2 Marks)

We load MNIST directly through Keras' built-in loader, which downloads the dataset
from its official source URL the first time it runs (no manual Kaggle download required).
We then convert it into a Pandas DataFrame, exactly like the CSV version from Kaggle
(`https://www.kaggle.com/datasets/oddrationale/mnist-in-csv`) would look — one row per image,
784 pixel columns + 1 label column.


In [ ]:
# Load MNIST via Keras (auto-downloads from Google's official MNIST mirror)
(X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

# Combine train+test into a single pool (we'll do our own 80/20 split per the assignment spec)
X_all = np.concatenate([X_train_raw, X_test_raw], axis=0)
y_all = np.concatenate([y_train_raw, y_test_raw], axis=0)

print("Total images:", X_all.shape[0])
print("Image dimensions:", X_all.shape[1], "x", X_all.shape[2])


In [ ]:
# Flatten each 28x28 image into 784 pixel columns and build a DataFrame (Pandas load step)
X_flat = X_all.reshape(X_all.shape[0], -1)
pixel_cols = [f'pixel{i}' for i in range(X_flat.shape[1])]

df = pd.DataFrame(X_flat, columns=pixel_cols)
df.insert(0, 'label', y_all)

df.head()


In [ ]:
# Input features and target variable
print("Input features: 784 columns (pixel0 ... pixel783), each representing one pixel intensity (0-255)")
print("Target variable: 'label' column, the digit (0-9) the image represents")


In [ ]:
# Dataset dimensions and summary
print("Dataset shape:", df.shape)
df.info()


In [ ]:
df.describe()


In [ ]:
# Display one sample handwritten digit
sample_idx = 0
sample_image = df.iloc[sample_idx, 1:].values.reshape(28, 28)
sample_label = df.iloc[sample_idx, 0]

plt.figure(figsize=(4, 4))
plt.imshow(sample_image, cmap='gray')
plt.title(f"Sample digit — Label: {sample_label}")
plt.axis('off')
plt.show()


## Task 2: Data Preprocessing (2 Marks)

In [ ]:
# Check for missing values
print("Total missing values in dataset:", df.isnull().sum().sum())


In [ ]:
# Separate features and target
X = df.drop('label', axis=1).values
y = df['label'].values

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# Normalize pixel values to range 0-1
X = X / 255.0
print("Min pixel value:", X.min(), " Max pixel value:", X.max())


In [ ]:
# Split into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


In [ ]:
# One-Hot Encode the target labels
y_train_cat = to_categorical(y_train, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print("y_train_cat shape:", y_train_cat.shape)
print("Example label:", y_train[0], "-> one-hot:", y_train_cat[0])


## Task 3: Model Development (3 Marks)

Architecture:
- Input Layer: 784 features
- Hidden Layer 1: 128 neurons, ReLU
- Hidden Layer 2: 64 neurons, ReLU
- Output Layer: 10 neurons, Softmax


In [ ]:
model = Sequential([
    Dense(128, activation='relu', input_shape=(784,)),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
# Train the model for 10 epochs
history = model.fit(
    X_train, y_train_cat,
    epochs=10,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)


In [ ]:
# Predict on the test dataset
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("Sample predictions:", y_pred[:10])
print("Actual labels:     ", y_test[:10])


## Task 4: Model Evaluation (2 Marks)

In [ ]:
# Test accuracy
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# Classification Report
print(classification_report(y_test, y_pred))


In [ ]:
# Accuracy vs Epoch
plt.figure(figsize=(7, 5))
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Loss vs Epoch
plt.figure(figsize=(7, 5))
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss vs Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()


### Observations

*(Fill these in based on your actual run — here are prompts to guide you)*

1. **Overall performance:** State the final test accuracy and whether it matches training accuracy (check for overfitting/underfitting).
2. **Convergence:** Note how quickly accuracy rose and loss dropped across the 10 epochs — did it plateau?
3. **Confusion patterns:** Look at the confusion matrix — which digit pairs does the model confuse most often (commonly digits with similar shapes, e.g. 4/9, 3/5, 7/1)?
4. **Precision/Recall:** From the classification report, note if any digit class has noticeably lower precision or recall than the rest.


## Task 5: Conclusion (1 Mark)

*(Write your 100–150 word conclusion here, covering: key findings, importance of hidden layers,
one advantage of Deep Learning over traditional ML, and one limitation of ANN.)*

**Draft to edit with your actual results:**

> This project implemented an Artificial Neural Network to classify handwritten digits from the MNIST
> dataset, achieving a test accuracy of **[insert your accuracy]**. The model's two hidden layers (128 and
> 64 neurons with ReLU activation) allowed it to learn increasingly abstract, non-linear representations of
> pixel patterns — without them, the network would reduce to a simple linear classifier incapable of
> capturing the complex shapes that distinguish digits. One key advantage of Deep Learning over traditional
> Machine Learning is its ability to automatically learn features directly from raw pixel data, removing the
> need for manual feature engineering. However, a notable limitation of a plain ANN is that it treats each
> pixel independently and ignores spatial relationships between neighboring pixels, which convolutional
> architectures (CNNs) are better suited to exploit. Overall, the model performed well, though some
> confusion persisted between visually similar digits.
